# 从消元提取一个判据

先比较 M4=[[1,2],[2,4]] 与 M5=[[1,2],[2,5]]，看最后一条方程是否保留自由变量。随后再看图与面积，不先背 ad-bc。

依赖 NumPy、Matplotlib。从新内核按顺序运行；图中使用英文，避免中文字体缺失。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
plt.rcParams.update({"font.family":"DejaVu Sans","font.size":11})
def matrix(t):
    return np.array([[1.,2.],[2.,float(t)]])
M4,M5 = matrix(4),matrix(5)


## 1. 同一步消元，为什么答案不同？

先手算第二行减第一行的2倍。哪组能让 z2 自由选择？

对于 M4，先猜一个非零零空间向量；对于 M5，解释为什么只剩零向量。

In [ ]:
for t in [4,5]:
    M = matrix(t)
    U = M.copy()
    U[1] -= 2*U[0]
    print("t =",t,"after elimination:\n",U)
    print("test input (-2,1):",M @ np.array([-2.,1.]))
assert np.allclose(M4 @ np.array([-2.,1.]),0)
assert not np.allclose(M5 @ np.array([-2.,1.]),0)


## 2. 把最后一个系数留成参数

消元得到 (t-4)z2=0。先预测 t=4、4.5、3 时，是直线零空间还是只有零向量。

只有 t=4 才保留自由变量；t 接近4不等于已经等于4。

In [ ]:
t = 4.5  # Also try 4 or 3.
M = matrix(t)
criterion = t-4
manual = M[0,0]*M[1,1]-M[0,1]*M[1,0]
print("last coefficient:",criterion,"ad-bc:",manual)
assert np.isclose(criterion,manual)
assert np.isclose(manual,np.linalg.det(M))


## 3. 按列看：还能撑出平面吗？

先比较两列是否为倍数关系。原来的第二列是第一列的2倍，改为 (2,5) 后呢？

图中四个顶点是原点、第一列、两列之和、第二列。它们对应单位正方形的四个顶点。

In [ ]:
square = np.array([[0.,0.],[1.,0.],[1.,1.],[0.,1.]])
fig,axes = plt.subplots(1,2,figsize=(8,6),dpi=150)
for ax,t in zip(axes,[4,5]):
    M = matrix(t)
    output = square @ M.T
    ax.add_patch(Polygon(output,closed=True,color="#359969",alpha=0.2))
    ax.plot(*np.vstack([output,output[0]]).T,color="#359969",lw=2)
    for index,color in [(1,"#d97922"),(0,"#2878b5")]:
        column = M[:,index]
        ax.annotate("",xy=column,xytext=(0,0),
                    arrowprops={"arrowstyle":"->","color":color,"lw":2.5})
    ax.set(xlim=(-0.5,3.8),ylim=(-0.5,7.8),aspect="equal",
           xlabel="Output 1",ylabel="Output 2",title=f"t={t}: det={t-4}")
    ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()


## 4. 非零变化与不同输入

取输入 (1,0)，再加上 (-2,1)。先预测两矩阵分别能否分清它们。

行列式为零时丢失的是什么？不是每个输入都被映成零，而是有非零变化不影响输出。

In [ ]:
first = np.array([1.,0.])
second = first+np.array([-2.,1.])
for M in [M4,M5]:
    print("matrix:\n",M,"first output:",M @ first,"second output:",M @ second)
assert np.allclose(M4 @ first,M4 @ second)
assert not np.allclose(M5 @ first,M5 @ second)


## 5. 上下剪切，怎样把斜底边变水平？

用 A=[[3,1],[1,3]]。手算每个点的 y 坐标减去 x/3。第一列变成什么？第二列的新高度是多少？

上下剪切不改变面积。变形以后可以直接用底乘高。

In [ ]:
A = np.array([[3.,1.],[1.,3.]])
H = np.array([[1.,0.],[-1/3,1.]])
T = H @ A
print("after shear:\n",T)
assert np.allclose(T,[[3,1],[0,8/3]])
base,height = 3.,8/3
print("base times height:",base*height,"ad-bc:",3*3-1*1)
assert np.isclose(base*height,8)
fig,axes = plt.subplots(1,2,figsize=(10,5),dpi=150)
for ax,M,title in [(axes[0],A,"Slanted base"),(axes[1],T,"Same area, horizontal base")]:
    output = square @ M.T
    ax.add_patch(Polygon(output,closed=True,color="#359969",alpha=0.2))
    ax.plot(*np.vstack([output,output[0]]).T,color="#359969",lw=2)
    for index,color in [(0,"#2878b5"),(1,"#d97922")]:
        ax.annotate("",xy=M[:,index],xytext=(0,0),
                    arrowprops={"arrowstyle":"->","color":color,"lw":2.5})
    ax.set(xlim=(-0.5,4.8),ylim=(-0.5,4.8),aspect="equal",
           xlabel="Output 1",ylabel="Output 2",title=title)
    ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()


## 6. 面积不为负，为什么行列式可以为负？

交换 A 的两列。先预测它们撑起的区域会不会变，再算 ad-bc。

交换坐标的矩阵行列式为-1，但交换两次即可恢复输入。负号不代表不可逆。

In [ ]:
swapped = A[:,[1,0]]
print("original determinant:",np.linalg.det(A))
print("swapped determinant:",np.linalg.det(swapped))
assert np.isclose(np.linalg.det(A),8)
assert np.isclose(np.linalg.det(swapped),-8)
swap = np.array([[0.,1.],[1.,0.]])
assert np.isclose(np.linalg.det(swap),-1)
assert np.allclose(swap @ swap,np.eye(2))


## 7. 行列式为零，不等于所有目标都无解

对 M4，输出第二项必须为第一项的2倍。先判断 (1,2) 和 (1,3) 哪个可达。

对 M5，两条方程相减得到 x2=q-2p，再由第一条求 x1。

In [ ]:
reachable = np.array([1.,2.])
impossible = np.array([1.,3.])
assert np.allclose(M4 @ np.array([1.,0.]),reachable)
print("M4 target (1,2), consistency residual:",reachable[1]-2*reachable[0])
print("M4 target (1,3), consistency residual:",impossible[1]-2*impossible[0])
p,q = impossible
x2 = q-2*p
x1 = p-2*x2
solution = np.array([x1,x2])
print("M5 input for (1,3):",solution,"output:",M5 @ solution)
assert np.allclose(M5 @ solution,impossible)


## 8. 回到特征值问题

先手算 det(A-lambda I)=(3-lambda)^2-1。

lambda=2、4、3 时，新矩阵分别是否有非零零空间？原矩阵 A 本身呢？

In [ ]:
for value in [2.,4.,3.,0.]:
    B = A-value*np.eye(2)
    determinant = (3-value)**2-1
    print("lambda:",value,"det:",determinant,"shifted matrix:\n",B)
    assert np.isclose(determinant,np.linalg.det(B))
assert np.allclose((A-4*np.eye(2)) @ np.array([1.,1.]),0)
assert np.allclose((A-2*np.eye(2)) @ np.array([1.,-1.]),0)
assert np.linalg.matrix_rank(A)==2


## 9. 补充核对：面积倍数相乘不等于矩阵可交换

剪切后拉伸与拉伸后剪切，形状可能不同，面积倍数却相同。

先算两个矩阵乘积，比较输出，然后核对行列式。

In [ ]:
shear = np.array([[1.,1.],[0.,1.]])
stretch = np.diag([1.,2.])
assert not np.allclose(shear @ stretch,stretch @ shear)
assert np.isclose(np.linalg.det(shear @ stretch),2)
assert np.isclose(np.linalg.det(stretch @ shear),2)
assert np.isclose(np.linalg.det(stretch @ shear),
                  np.linalg.det(stretch)*np.linalg.det(shear))


## 合上实验，自己重建

1. 为什么 t-4 能判断是否保留自由变量？
2. 在不除以 a 的条件下，怎样由一般方程得到 ad-bc？
3. 为什么行列式为零既可能无解，也可能无穷多个解，但不会有唯一解？这里限定方阵。
4. 为什么面积解释取绝对值？
5. 为什么找特征值时让移位矩阵不可逆，而不是让原矩阵不可逆？

参考：[MIT 18.06 第十八讲](https://ocw.mit.edu/courses/18-06-linear-algebra-spring-2010/04979ab05f4f27e621e1e26ece4586fd_MIT18_06S10_L18.pdf)。